In [ ]:
import transformers
#注意transformers的版本要新一点，否则不能识别新的LLM
from transformers import AutoModelForCausalLM, AutoTokenizer,BitsAndBytesConfig
import torch
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "3,4,5"#设置使用服务器中哪一个gpu，否则会占用所有的GPU

In [ ]:
model_id = "/data/share_weight/Meta-Llama-3.1-70B-Instruct"
quantization_config = BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_compute_dtype=torch.float16,bnb_4bit_use_double_quant=True,bnb_4bit_quant_type="nf4")
quantized_model = AutoModelForCausalLM.from_pretrained(
    model_id, device_map="auto", quantization_config=quantization_config)
tokenizer = AutoTokenizer.from_pretrained(model_id)
pipeline = transformers.pipeline("text-generation", model=quantized_model, tokenizer=tokenizer,max_new_tokens=1024,pad_token_id=128001)

测试prompt

In [ ]:
input_text = open(
    os.path.join("./prompt/ett_type_filter.md"),
    'r', encoding='utf-8'
).read()
q = "what did james k polk do before he was president?"
t = "James K. Polk"
e = "{\"James K. Polk\" : \"['type.object', 'government.us_president', 'people.appointer', 'book.author', 'common.topic', 'kg.object_profile', 'people.deceased_person', 'people.person', 'government.politician', 'location.location', 'base.ranker', 'organization.organization_founder', 'symbols.name_source', 'book.book_subject', 'authority.daylife', 'common.identity', 'base.uspolitician', 'government.u_s_congressperson', 'government.political_appointer', 'type.type', 'book.written_work', 'people.marriage', 'education.education', 'government.government_position_held', 'government.political_party_tenure', 'organization.organization', 'people.place_lived', 'people.place_of_interment', 'people.profession', 'dataworld.gardening_hint', 'people.sibling_relationship', 'symbols.namesake', 'book.literary_series', 'government.election_campaign', 'common.image', 'government.us_vice_president', 'people.cause_of_death', 'government.election', 'media_common.quotation', 'people.appointment', 'people.ethnicity', 'freebase.list_entry']\"}"
input_text = input_text + f"Question: {q}\n" + f"Topic Entity: {t}\n" + f"Entity Types: {e}\n" + "Most Relevant Entity Types:"

In [ ]:
input_text = open(
    os.path.join("./prompt/test_exit.md"),
    'r', encoding='utf-8'
).read()
q = "who is keyshia cole dad?"
r = "Keyshia_Cole -> people.person.parent"
t = '[("Keyshia_Cole", "people.person.parents", "Sal Gibson"), ("Keyshia Cole", "people.person.parents", "Francine Lons"), ("Keyshia Cole", "people.person.parents", "Yvonne Cole"), ("Keyshia Cole", "people.person.parents", "Leon Cole"), ("Keyshia Cole", "people.person.parents", "Daniel Hiram Gibson Jr.")]'
e = "Keyshia_Cole, Keyshia_Cole_parents"
input_text = input_text.format(q, r, t, e)

In [ ]:
input_text = open(
    os.path.join("./prompt/question_constrain.md"),
    'r', encoding='utf-8'
).read()
q = "what did the islamic people believe in?"
input_text += q

In [ ]:
messages = [{"role": "user", "content": input_text}]
prompt = pipeline.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
terminators = [pipeline.tokenizer.eos_token_id, pipeline.tokenizer.convert_tokens_to_ids("<|eot_id|>")]
outputs = pipeline(prompt, eos_token_id=terminators, do_sample=True)
pred_solution="".join(outputs[0]["generated_text"][len(prompt):])
print(pred_solution)

相似度比较

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "4"
enc_model = SentenceTransformer("../all-MiniLM-L6-v2")
def similar_relation_select(cand_relations, init_relation):
    #cand_encode_list = np.asarray([enc_model.encode(r) for r in cand_relations])
    cand_encode_list = np.asarray(enc_model.encode(cand_relations))
    init_encode = np.asarray([enc_model.encode(init_relation)])
    d = len(cand_encode_list[0])
    index = faiss.IndexFlatL2(d)
    index.add(cand_encode_list)
    D, I = index.search(init_encode, 5)
    index = I
    selected_relation = [cand_relations[i] for i in I[0]]
    return selected_relation

cand_relations = ['location.country', 'cvg.computer_game_region', 'common.topic', 'kg.object_profile', 'location.statistical_region', 'base.aareas.schema.administrative_area', 'sports.sport_country', 'wikipedia', 'base.dspl.world_bank.wdi_gdf', 'base.locations.countries', 'biology.breed_origin', 'location.location', 'olympics.olympic_participating_country', 'type.object']
init_relation = "what countries are part of the uk?"
'''for _ in range(9):
    cand_relations += cand_relations'''
print(similar_relation_select(cand_relations, init_relation))
print(len(cand_relations))

实体查询

In [2]:
from SPARQLWrapper import SPARQLWrapper, JSON
sparql_head_relations = """\nPREFIX ns: <http://rdf.freebase.com/ns/>\nSELECT ?relation\nWHERE {\n  ns:%s ?relation ?x .\n}"""
sparql_tail_relations = """\nPREFIX ns: <http://rdf.freebase.com/ns/>\nSELECT ?relation\nWHERE {\n  ?x ?relation ns:%s .\n}"""

sparql_head_ett_values = """\nPREFIX ns: <http://rdf.freebase.com/ns/>\nSELECT ?head\nWHERE {\n  ?head ?x ns:%s .\n}"""
sparql_tail_ett_values = """\nPREFIX ns: <http://rdf.freebase.com/ns/>\nSELECT ?tail ?x\nWHERE {\n  ns:%s ?x ?tail .\n}"""
sparql_tail_entities_extract = """PREFIX ns: <http://rdf.freebase.com/ns/>\nSELECT ?Entity\nWHERE {\nns:%s ns:%s ?Entity .\n}""" 
sparql_id = """PREFIX ns: <http://rdf.freebase.com/ns/>\nSELECT DISTINCT ?tailEntity\nWHERE {\n  {\n    ?entity ns:type.object.name ?tailEntity .\n    FILTER(?entity = ns:%s)\n  }\n  UNION\n  {\n    ?entity <http://www.w3.org/2002/07/owl#sameAs> ?tailEntity .\n    FILTER(?entity = ns:%s)\n  }\nFILTER (LANG(?tailEntity) = "en")\n}"""
desc = """\nPREFIX ns: <http://rdf.freebase.com/ns/>\nSELECT ?tail\nWHERE {\n  ns:%s ns:common.topic.description ?tail .\nFILTER (LANG(?tail) = "en")\n}"""

mid = ["m.0c9_j12", "m.0cb_npm", "m.049x6zj", "m.049x6_k", "m.04hzvwh", "m.049x6zw", "m.049x6_6", "m.049x6_x", "m.0c5f3d3", "m.049x6yv", "m.049x6z5", "m.0c5f1pl", "m.0vsc2y3"]
rel1 = sparql_head_relations % 'm.0j4jq57'# 'm.01_2n' 'm.010vz' 'm.1h3m1x5' m.042f1 m.03_r3 m.01dw03
rel2 = sparql_tail_relations % 'm.042f1'
ett1 = sparql_tail_entities_extract % ("m.03rjj", "location.country.currency_used") #organization.organization.founders
ett2 = sparql_tail_entities_extract % ("m.0cb_npm", "government.government_position_held.office_holder")
ett3 = sparql_tail_entities_extract % ("m.0cb_npm", "government.government_position_held.from")
type_name_1 = sparql_id % ("m.02l6h", "m.02l6h")
type_name_2 = sparql_id % ("m.05wh0sh", "m.05wh0sh")
type_name_3 = sparql_id % ("m.03k5sq", "m.03k5sq")
#q_list = [type_name]
q_list = [rel1, type_name_3]
q_list = [ett1, type_name_1, ett2, type_name_2, ett3]


for query in q_list:
    sparql = SPARQLWrapper("http://localhost:3003/sparql")
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    results = sparql.queryAndConvert()
    #print(results["results"]["bindings"])
    for i in results["results"]["bindings"]:
        if 'Entity' in i:
            print(i['Entity']['value']) 
        if 'tailEntity' in i:
            print(i['tailEntity']['value'])
        if 'relation' in i:
            print(i['relation']['value'])
            print(i['x']['value'])


http://rdf.freebase.com/ns/m.02l6h
Euro
http://rdf.freebase.com/ns/m.05wh0sh
Vladimir Lenin
1917-08:00


三元组搜索

In [3]:
from SPARQLWrapper import SPARQLWrapper, JSON
sparql_head_relations = """\nPREFIX ns: <http://rdf.freebase.com/ns/>\nSELECT ?relation\nWHERE {\n  ns:%s ?relation ?x .\n}"""
sparql_tail_relations = """\nPREFIX ns: <http://rdf.freebase.com/ns/>\nSELECT ?relation\nWHERE {\n  ?x ?relation ns:%s .\n}"""

sparql_tail_entities_extract = """PREFIX ns: <http://rdf.freebase.com/ns/>\nSELECT ?tailEntity\nWHERE {\nns:%s ns:%s ?tailEntity .\n}""" 
sparql_tail_entities_extract_with_type = """PREFIX ns: <http://rdf.freebase.com/ns/>\nPREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>\nSELECT ?tailEntity\nWHERE {\nns:%s rdf:type ?tailEntity .\n}""" 
sparql_head_entities_extract = """PREFIX ns: <http://rdf.freebase.com/ns/>\nSELECT ?tailEntity\nWHERE {\n?tailEntity ns:%s ns:%s  .\n}"""

sparql_id = """PREFIX ns: <http://rdf.freebase.com/ns/>\nSELECT DISTINCT ?tailEntity\nWHERE {\n  {\n    ?entity ns:type.object.name ?tailEntity .\n    FILTER(?entity = ns:%s)\n  }\n  UNION\n  {\n    ?entity <http://www.w3.org/2002/07/owl#sameAs> ?tailEntity .\n    FILTER(?entity = ns:%s)\n  }\nFILTER (LANG(?tailEntity) = "en")\n}"""
desc = """\nPREFIX ns: <http://rdf.freebase.com/ns/>\nSELECT ?tail\nWHERE {\n  ns:%s ns:common.topic.description ?tail .\nFILTER (LANG(?tail) = "en")\n}"""

SPARQLPATH = "http://localhost:3002/sparql"

def execute_sparql(sparql_txt):
    sparql_txt = 'PREFIX : <http://rdf.freebase.com/ns/>\n'+sparql_txt
    try:
        sparql = SPARQLWrapper(SPARQLPATH)
        sparql.setQuery(sparql_txt)
        sparql.setReturnFormat(JSON)
        sparql.addExtraURITag("timeout", "10000")
        results = sparql.query().convert()

        res = []
        for x in results["results"]["bindings"]:
            res_item = {}
            for k, v in x.items():
                res_item[k] = v['value']
            res.append(res_item)
        return res
    except Exception as e:
        print(e)
        print("Freebase query error")
        print(sparql_txt)
        return []

def table_result_to_list(res):
    if len(res) == 0:
        return []
    else:
        key_list = res[0].keys()
        result = {}
        for key in key_list:
            result[key] = list(set([item[key] for item in res]))
        return result

def entity_search(entity, relation, head=True):
    if head:
        if "http://www.w3.org/1999/02/22-rdf-syntax-ns#type" in relation:
            tail_entities_extract = sparql_tail_entities_extract_with_type% (entity)
            entities = table_result_to_list(execute_sparql(tail_entities_extract))
        else:
            tail_entities_extract = sparql_tail_entities_extract% (entity, relation)
            entities = table_result_to_list(execute_sparql(tail_entities_extract))
    else:
        head_entities_extract = sparql_head_entities_extract% (relation, entity)
        entities = table_result_to_list(execute_sparql(head_entities_extract))

    if entities != []:
        entities = entities['tailEntity']
        #entities = [x.replace("http://rdf.freebase.com/ns/", "") for x in entities if 'http://rdf.freebase.com/ns' in x]
        entities = [x.split('/')[-1] for x in entities]

    new_entity = [entity for entity in entities]

    return new_entity

def id2entity_name_or_type_en(entity_id):
    if entity_id.startswith("m.") == False and entity_id.startswith("g.") == False:
        return entity_id
    
    sparql_query = sparql_id % (entity_id, entity_id)
    sparql = SPARQLWrapper(SPARQLPATH)
    sparql.setQuery(sparql_query)
    sparql.setReturnFormat(JSON)
    try:
        results = sparql.queryAndConvert()

        if len(results["results"]["bindings"])==0:
            return entity_id
        else:
            for lines in results["results"]["bindings"]:
                if lines['tailEntity']['xml:lang']=='en':
                    return lines['tailEntity']['value']
                
            return results["results"]["bindings"][0]['tailEntity']['value']
    except:
        return entity_id

def triple(ett, ett_id, rel, head=True):
    results = entity_search(ett_id, rel, head)
    tp_list = []
    for i in results:
        if head == True:
            tp_list.append("({}, {}, {}({}))".format(ett, rel, id2entity_name_or_type_en(i), i))
        else:
            tp_list.append("({}({}), {}, {})".format(id2entity_name_or_type_en(i), i, rel, ett))
    return tp_list

ett = "DM"         # Destin–Fort Walton Beach Airport
ett_id = 'm.027rn' # m.01q6d0
sparql = SPARQLWrapper("http://localhost:3003/sparql")
sparql.setQuery(sparql_head_relations%ett_id)
sparql.setReturnFormat(JSON)
results = sparql.queryAndConvert()
print(len(results["results"]["bindings"]))
relation = list(set([i['relation']['value'].split('/')[-1] for i in results["results"]["bindings"]]))
print(relation)
target_relation = [i for i in relation if "computer" in i]
print("target relation:", target_relation)
triples_list = []
for i in relation:
    if i.startswith('type') == True or i.startswith('kg') == True or i.startswith('user') == True or i.startswith('common') == True or i.startswith('wikipedia') == True or i.startswith('freebase') == True or '#' in i:
        continue
    if 'continent' in i:
        triples_list.extend(triple(ett, ett_id, i, head=True))

sparql = SPARQLWrapper("http://localhost:3003/sparql")
sparql.setQuery(sparql_tail_relations%ett_id)
sparql.setReturnFormat(JSON)
results = sparql.queryAndConvert()
print(len(results["results"]["bindings"]))
relation = list(set([i['relation']['value'].split('/')[-1] for i in results["results"]["bindings"]]))
print(relation)
target_relation = [i for i in relation if "computer" in i]
print("target relation:", target_relation)
for i in relation:
    if i.startswith('type') == True or i.startswith('kg') == True or i.startswith('user') == True or i.startswith('common') == True or 'wikipedia' in i or 'freebase' in i == True or '#' in i:
        continue
    if 'continent' in i:
        triples_list.extend(triple(ett, ett_id, i, head=False))
print(', '.join(triples_list))



4056
['base.aareas.schema.administrative_area.subdividing_type', 'type.object.type', 'authority.iso.3166-1.alpha-3', 'wikipedia.zh-tw', 'wikipedia.ru', 'wikipedia.sl_title', 'location.country.form_of_government', 'wikipedia.el', 'location.statistical_region.co2_emissions_per_capita', 'location.country.iso3166_1_alpha2', 'location.statistical_region.net_migration', 'type.object.key', 'location.statistical_region.health_expenditure_as_percent_of_gdp', 'location.statistical_region.life_expectancy', 'location.statistical_region.population', 'location.statistical_region.gdp_real', 'wikipedia.vi', 'wikipedia.hu_title', 'location.country.official_language', 'location.country.gdp_nominal', 'type.object.name', 'wikipedia.es', 'wikipedia.lt', 'location.location.partially_contains', 'wikipedia.lv_id', 'location.statistical_region.internet_users_percent_population', 'location.location.partiallycontains', 'base.ontologies.ontology_instance.equivalent_instances', 'sports.sport_country.athletes', 'wi

测试KG搜索（type）

In [6]:
from SPARQLWrapper import SPARQLWrapper, JSON
import json
from tqdm import tqdm

q_type = """\nPREFIX ns: <http://rdf.freebase.com/ns/>\nSELECT ?type\nWHERE {\n  ns:%s rdf:type ?type .\n}\nLIMIT 100"""
query = q_type
m = "m.07ssc"
sparql = SPARQLWrapper("http://127.0.0.1:3003/sparql")
type_list = []
domain_list = []
sparql.setQuery(query%m)
sparql.setReturnFormat(JSON)
results = sparql.queryAndConvert()
#print(len(results["results"]["bindings"]))
for i in results["results"]["bindings"]:
    if "http://rdf.freebase.com/ns/" in i['type']['value'] and 'user' not in i['type']['value']:
        type_list.append(i['type']['value'].split('/')[-1])
        domain_list.append(i['type']['value'].split('/')[-1].split('.')[0])
    if 'base.aareas' in i['type']['value']:
        print(i['type']['value'].split('/')[-1])
print(type_list)

print(len(type_list))

sparql_head_relations = """\nPREFIX ns: <http://rdf.freebase.com/ns/>\nSELECT ?relation\nWHERE {\n  ns:%s ?relation ?x .\n}"""
sparql_tail_relations = """\nPREFIX ns: <http://rdf.freebase.com/ns/>\nSELECT ?relation\nWHERE {\n  ?x ?relation ns:%s .\n}"""
query = sparql_head_relations
m = "m.07ssc"
sparql = SPARQLWrapper("http://127.0.0.1:3003/sparql")
type_list = []
domain_list = []
sparql.setQuery(query%m)
sparql.setReturnFormat(JSON)
results = sparql.queryAndConvert()
#print(len(results["results"]["bindings"]))
for i in results["results"]["bindings"]:
    #if "http://rdf.freebase.com/ns/" in i['relation']['value']:
    r = i['relation']['value'].split('/')[-1]
    t = r.split('.')
    type_list.append(r[:-len(t[-1])-1])
type_list = list(set(type_list))
print(type_list)
print(len(type_list))      

base.aareas.schema.earth.sovereign_state
['base.aareas.schema.earth.sovereign_state', 'biology.breed_origin', 'sports.sport_country', 'base.events.geographical_scope', 'base.leicester.topic', 'base.locations.countries', 'base.jewlib.topic', 'base.type_ontology.abstract', 'base.type_ontology.agent', 'base.type_ontology.inanimate', 'common.topic', 'location.country', 'location.location', 'location.statistical_region', 'location.dated_location', 'business.business_location', 'business.employer', 'base.schemastaging.topic', 'base.tagit.concept', 'book.book_subject', 'organization.organization_founder', 'sports.sports_team_location', 'government.governmental_jurisdiction', 'base.petbreeds.topic', 'base.skosbase.topic', 'base.skosbase.vocabulary_equivalent_topic', 'organization.organization_scope', 'olympics.olympic_participating_country', 'base.charities.topic', 'tv.tv_location', 'base.prestatyn.topic', 'base.biblioness.bibs_location', 'base.popstra.sww_base', 'base.popstra.topic', 'periodi

测试搜索宾语谓词

In [4]:
from SPARQLWrapper import SPARQLWrapper, JSON
import json
from tqdm import tqdm

q_type = """\nPREFIX ns: <http://rdf.freebase.com/ns/>\nSELECT ?type\nWHERE {\n  ns:%s ns:type.property.expected_type ?type .\n}\nLIMIT 50"""
q_unique = """\nPREFIX ns: <http://rdf.freebase.com/ns/>\nSELECT ?type\nWHERE {\n  ns:%s ns:type.property.unique ?type .\n}\nLIMIT 50"""
query = q_type
m = "government.government_position_held.from"
sparql = SPARQLWrapper("http://127.0.0.1:3003/sparql")
type_list = []
domain_list = []
sparql.setQuery(query%m)
sparql.setReturnFormat(JSON)
results = sparql.queryAndConvert()
print(results["results"]["bindings"])

[{'type': {'type': 'uri', 'value': 'http://rdf.freebase.com/ns/type.datetime'}}]


实体名称搜索

In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSON
import json
from tqdm import tqdm
sparql_id = """PREFIX ns: <http://rdf.freebase.com/ns/>\nSELECT DISTINCT ?tailEntity\nWHERE {\n  {\n    ?entity ns:type.object.name ?tailEntity .\n    FILTER(?entity = ns:%s)\n  }\n  UNION\n  {\n    ?entity <http://www.w3.org/2002/07/owl#sameAs> ?tailEntity .\n    FILTER(?entity = ns:%s)\n  }\nFILTER (LANG(?tailEntity) = "en")\n}"""
sparql = SPARQLWrapper("http://localhost:3002/sparql")
type_list = []
domain_list = []
m = "m.05zppz"
sparql.setQuery(sparql_id % (m, m))
sparql.setReturnFormat(JSON)
results = sparql.queryAndConvert()
print(results["results"]["bindings"])


关系实例化

In [ ]:
import os, sys
from argparse import ArgumentParser
from tqdm import tqdm
import numpy as np
from config import LLM_BASE
import json
import random
sys.path.append(os.path.dirname(os.path.realpath(__file__)) + "/..")
from utils.utils import run_llm, get_timestamp, readjson
from utils.freebase_func import *
import time
from tqdm import tqdm
from utils import *
from config import *
from kg_instantiation import *
import tiktoken


relations_no_q = grounding_relations('type', topk=20)
print(relations_no_q)